In [14]:
# Cell 1 — CLEANUP: remove previously saved csv/json files so we start fresh
import os, glob, shutil

# directories we use
raw_dir = "raw_data"
graphs_dir = "graphs"

# create dirs if not exist
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(graphs_dir, exist_ok=True)

# delete CSVs/JSONs in raw_data and graphs
for pattern in [os.path.join(raw_dir, "*.csv"), os.path.join(raw_dir, "*.json"),
                os.path.join(graphs_dir, "*.json"), os.path.join(graphs_dir, "*.csv")]:
    for f in glob.glob(pattern):
        try:
            os.remove(f)
        except Exception as e:
            print("Could not remove", f, ":", e)

print("Cleaned raw_data/ and graphs/ of previous CSV/JSON files (if any).")


Cleaned raw_data/ and graphs/ of previous CSV/JSON files (if any).


In [15]:
# Cell 2 — Imports and folder setup
import os
from datetime import datetime, timedelta
import random
import hashlib
import uuid
import numpy as np
import pandas as pd

# Ensure folders exist
os.makedirs("raw_data", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("graphs", exist_ok=True)

print("Folders ensured: raw_data/, data/, graphs/")


Folders ensured: raw_data/, data/, graphs/


In [16]:
# Cell 3 — Parameters and helper functions
NUM_ACCOUNTS = 5000        # total accounts to simulate
NUM_TRANSACTIONS = 20000   # baseline random transactions
NUM_MULES = 200            # number of mule accounts
START_DATE = datetime.now() - timedelta(days=30)
END_DATE = datetime.now()

def make_vpa(i):
    return f"user{i:05d}@bank"

def make_device_id():
    return hashlib.sha1(uuid.uuid4().bytes).hexdigest()[:12]

def random_timestamp(start, end):
    span = (end - start).total_seconds()
    return start + timedelta(seconds=random.randint(0, int(span)))

np.random.seed(42)
random.seed(42)

print("Parameters set: accounts", NUM_ACCOUNTS, "transactions baseline", NUM_TRANSACTIONS, "mules", NUM_MULES)


Parameters set: accounts 5000 transactions baseline 20000 mules 200


In [17]:
# Cell 4 — Create accounts_df
accounts = []
for i in range(NUM_ACCOUNTS):
    account_id = f"A{i:06d}"
    vpa = make_vpa(i)
    # spread created_at over ~365 days (so some older/newer)
    created_at = START_DATE - timedelta(days=random.randint(0, 335))
    device_id = make_device_id()
    accounts.append({
        "account_id": account_id,
        "vpa": vpa,
        "created_at": created_at,
        "device_id": device_id
    })

accounts_df = pd.DataFrame(accounts)
print("accounts_df created with rows:", len(accounts_df))
accounts_df.head()


accounts_df created with rows: 5000


,account_id,vpa,created_at,device_id
0,A000000,user00000@bank,2024-11-27 22:42:57.608166,f84dfbd5f35d
1,A000001,user00001@bank,2025-08-24 22:42:57.608166,d974d045d8d6
2,A000002,user00002@bank,2025-10-08 22:42:57.608166,ba92f09f6014
3,A000003,user00003@bank,2025-06-02 22:42:57.608166,559e0ad75e57
4,A000004,user00004@bank,2025-06-17 22:42:57.608166,90dfe8262a29


In [18]:
# Cell 5 — Mark mule accounts and simulate device reuse among them
mule_ids = set(accounts_df.sample(NUM_MULES, random_state=123).account_id.tolist())
accounts_df["is_mule"] = accounts_df["account_id"].isin(mule_ids).astype(int)

# Simulate device reuse: group mule accounts into small groups sharing a device
mule_list = list(mule_ids)
for i in range(0, len(mule_list), 5):
    group = mule_list[i:i+5]
    if len(group) > 1:
        shared_device = make_device_id()
        accounts_df.loc[accounts_df["account_id"].isin(group), "device_id"] = shared_device

print("Marked mule accounts and applied device reuse to some groups.")
accounts_df.is_mule.value_counts()


Marked mule accounts and applied device reuse to some groups.


is_mule
0    4800
1     200
Name: count, dtype: int64

In [19]:
# Cell 6 — Generate baseline random transactions
txs = []
for t in range(NUM_TRANSACTIONS):
    sender = accounts_df.sample(1).account_id.values[0]
    receiver = accounts_df.sample(1).account_id.values[0]
    while receiver == sender:
        receiver = accounts_df.sample(1).account_id.values[0]
    ts = random_timestamp(START_DATE, END_DATE)
    amount = round(float(np.random.exponential(scale=1500)) + 10, 2)
    txs.append({
        "tx_id": f"T{t:07d}",
        "timestamp": ts,
        "sender": sender,
        "receiver": receiver,
        "amount": amount
    })

transactions_df = pd.DataFrame(txs).sort_values("timestamp").reset_index(drop=True)
print("Baseline transactions created:", len(transactions_df))
transactions_df.head()


Baseline transactions created: 20000


,tx_id,timestamp,sender,receiver,amount
0,T0011450,2025-10-20 22:43:36.608166,A003020,A004120,925.18
1,T0007410,2025-10-20 22:45:26.608166,A002906,A002865,2714.12
2,T0016049,2025-10-20 22:47:12.608166,A003892,A001515,703.90
3,T0005538,2025-10-20 22:48:55.608166,A000179,A004141,533.90
4,T0000666,2025-10-20 22:49:24.608166,A003145,A002498,1824.11


In [20]:
# Cell 7 — Inject mule-like transactions (many inbound senders, quick forwarding)
extra_txs = []
extra_tx_counter = NUM_TRANSACTIONS

non_mule = accounts_df[accounts_df["is_mule"] == 0].account_id.tolist()
mules = accounts_df[accounts_df["is_mule"] == 1].account_id.tolist()

for mule in mules:
    # many inbound senders within short window
    num_inbound = random.randint(10, 30)
    inbound_senders = random.sample(non_mule, num_inbound)
    base_time = random_timestamp(START_DATE, END_DATE)
    inbound_times = []
    for s in inbound_senders:
        ts = base_time + timedelta(minutes=random.randint(0, 180))
        amt = round(max(50, np.random.exponential(scale=2000)), 2)
        extra_txs.append({
            "tx_id": f"T{extra_tx_counter:07d}",
            "timestamp": ts,
            "sender": s,
            "receiver": mule,
            "amount": amt
        })
        inbound_times.append(ts)
        extra_tx_counter += 1

    # fast forwarding: mule forwards to other mule accounts shortly after inbound
    num_forwards = max(1, int(0.7 * num_inbound))
    forward_targets = [m for m in mules if m != mule]
    if forward_targets:
        forward_targets = random.sample(forward_targets, min(num_forwards, len(forward_targets)))
        for j in range(num_forwards):
            ts = inbound_times[j % len(inbound_times)] + timedelta(minutes=random.randint(1, 60))
            tgt = forward_targets[j % len(forward_targets)]
            amt = round(max(20, np.random.exponential(scale=1500)), 2)
            extra_txs.append({
                "tx_id": f"T{extra_tx_counter:07d}",
                "timestamp": ts,
                "sender": mule,
                "receiver": tgt,
                "amount": amt
            })
            extra_tx_counter += 1

print("Injected mule-style inbound and forwarding transactions. Extra txs:", len(extra_txs))


Injected mule-style inbound and forwarding transactions. Extra txs: 6553


In [21]:
# Cell 8 — Add circular flows among mule subsets and bursts after dormancy
# Circular rings
num_rings = max(5, NUM_MULES // 20)
for r in range(num_rings):
    ring_size = random.randint(3, 8)
    ring_members = random.sample(mules, ring_size)
    start_time = random_timestamp(START_DATE, END_DATE)
    for i in range(ring_size):
        sender = ring_members[i]
        receiver = ring_members[(i + 1) % ring_size]
        ts = start_time + timedelta(minutes=i * 5)
        amt = round(100 + np.random.rand() * 2000, 2)
        extra_txs.append({
            "tx_id": f"T{extra_tx_counter:07d}",
            "timestamp": ts,
            "sender": sender,
            "receiver": receiver,
            "amount": amt
        })
        extra_tx_counter += 1

# Bursts after dormancy for some mules
burst_mules = random.sample(mules, k=max(10, NUM_MULES // 10))
for mule in burst_mules:
    dormant_end = START_DATE + timedelta(days=random.randint(15, 25))
    burst_start = dormant_end + timedelta(hours=random.randint(1, 48))
    num_inbound = random.randint(5, 20)
    inbound_senders = random.sample(non_mule, num_inbound)
    for s in inbound_senders:
        ts = burst_start + timedelta(minutes=random.randint(0, 240))
        amt = round(max(50, np.random.exponential(scale=1200)), 2)
        extra_txs.append({
            "tx_id": f"T{extra_tx_counter:07d}",
            "timestamp": ts,
            "sender": s,
            "receiver": mule,
            "amount": amt
        })
        extra_tx_counter += 1
    for k in range(int(0.8 * num_inbound)):
        ts = burst_start + timedelta(minutes=random.randint(5, 360))
        tgt = random.choice([m for m in mules if m != mule])
        amt = round(max(20, np.random.exponential(scale=900)), 2)
        extra_txs.append({
            "tx_id": f"T{extra_tx_counter:07d}",
            "timestamp": ts,
            "sender": mule,
            "receiver": tgt,
            "amount": amt
        })
        extra_tx_counter += 1

print("Added circular rings and dormancy bursts. Total extra_txs now:", len(extra_txs))


Added circular rings and dormancy bursts. Total extra_txs now: 7048


In [22]:
# Cell 9 — Merge baseline and injected transactions; save CSVs
extra_df = pd.DataFrame(extra_txs)
all_txs = pd.concat([transactions_df, extra_df], ignore_index=True)
all_txs = all_txs.sort_values("timestamp").reset_index(drop=True)

print("Total transactions after merge:", len(all_txs))
print("Transaction time range:", all_txs.timestamp.min(), "→", all_txs.timestamp.max())
print("Unique senders:", all_txs.sender.nunique(), "Unique receivers:", all_txs.receiver.nunique())

# Save CSVs
accounts_df.to_csv("raw_data/accounts_simulated.csv", index=False)
all_txs.to_csv("raw_data/simulated_transactions.csv", index=False)
labels_df = accounts_df[["account_id", "vpa", "device_id", "is_mule"]].copy()
labels_df.to_csv("raw_data/account_labels.csv", index=False)

# save a smaller sample for quick dev
all_txs.sample(min(10000, len(all_txs)), random_state=1).to_csv("raw_data/simulated_transactions_sample.csv", index=False)

print("Saved: raw_data/accounts_simulated.csv, raw_data/simulated_transactions.csv, raw_data/account_labels.csv")


Total transactions after merge: 27048
Transaction time range: 2025-10-20 22:43:36.608166 → 2025-11-19 22:42:40.608166
Unique senders: 4963 Unique receivers: 4911
Saved: raw_data/accounts_simulated.csv, raw_data/simulated_transactions.csv, raw_data/account_labels.csv


In [23]:
# Cell 10 — Create ego-network snapshot for one mule and save JSON
import networkx as nx, json

example_mule = random.choice(mules)
recent_window = all_txs[all_txs.timestamp >= (END_DATE - timedelta(days=7))]

G = nx.DiGraph()
for _, r in recent_window.iterrows():
    G.add_edge(r["sender"], r["receiver"], amount=float(r["amount"]), ts=str(r["timestamp"]))

# Use undirected ego graph for neighborhood visualizations
ego = nx.ego_graph(G.to_undirected(), example_mule, radius=2)

ego_data = {
    "center": example_mule,
    "nodes": [{"id": n, "is_mule": 1 if n in mules else 0} for n in ego.nodes()],
    "edges": [{"source": u, "target": v} for u, v in ego.edges()]
}

with open(f"graphs/ego_example_{example_mule}.json", "w") as f:
    json.dump(ego_data, f)

print("Saved ego-network snapshot:", f"graphs/ego_example_{example_mule}.json")


Saved ego-network snapshot: graphs/ego_example_A002277.json


In [24]:
# Cell 11 — Quick summaries and validation checks
txs = all_txs.copy()
txs = txs.merge(labels_df.rename(columns={"account_id":"sender", "is_mule":"sender_is_mule"}), on="sender", how="left")
txs = txs.merge(labels_df.rename(columns={"account_id":"receiver", "is_mule":"receiver_is_mule"}), on="receiver", how="left")

summary = {
    "total_transactions": len(txs),
    "tx_from_mule": int(txs.sender_is_mule.sum()),
    "tx_to_mule": int(txs.receiver_is_mule.sum()),
    "unique_mule_accounts_seen": len(set(txs[txs.sender_is_mule==1].sender) | set(txs[txs.receiver_is_mule==1].receiver))
}
print("Summary:", summary)

# show top 8 transactions for manual check
txs.head(8)


Summary: {'total_transactions': 27048, 'tx_from_mule': 3736, 'tx_to_mule': 7895, 'unique_mule_accounts_seen': 200}


,tx_id,timestamp,sender,receiver,amount,vpa_x,device_id_x,sender_is_mule,vpa_y,device_id_y,receiver_is_mule
0,T0011450,2025-10-20 22:43:36.608166,A003020,A004120,925.18,user03020@bank,e68cdd5dd26a,0,user04120@bank,f632653ae62b,0
1,T0007410,2025-10-20 22:45:26.608166,A002906,A002865,2714.12,user02906@bank,3a31a02bb004,0,user02865@bank,3dee451a5846,0
2,T0016049,2025-10-20 22:47:12.608166,A003892,A001515,703.90,user03892@bank,8bdf212b8151,0,user01515@bank,4d405220eff5,0
3,T0005538,2025-10-20 22:48:55.608166,A000179,A004141,533.90,user00179@bank,679a63263f89,0,user04141@bank,a5ab20be10da,0
4,T0000666,2025-10-20 22:49:24.608166,A003145,A002498,1824.11,user03145@bank,9889f5b2c9f6,0,user02498@bank,a43ed2b9f8ed,0
5,T0013208,2025-10-20 22:49:26.608166,A002630,A002626,59.49,user02630@bank,86c640bb22db,0,user02626@bank,64d86a95acdf,0
6,T0001770,2025-10-20 22:55:50.608166,A000874,A004585,3332.77,user00874@bank,d1ba4735f4f6,0,user04585@bank,5227f9b9d2cd,0
7,T0000971,2025-10-20 22:56:37.608166,A004490,A004100,593.32,user04490@bank,e1403d81d98a,0,user04100@bank,ccd079b41478,0
